# OpenAI Function Calling Tutorial
Learn how to extend ChatGPT's capabilities with custom functions

## What is Function Calling?
- Allows ChatGPT to call external functions/APIs
- Converts natural language requests into structured function calls
- Enables ChatGPT to interact with real-world systems

## What we'll cover:
1. Basic function calling concepts
2. Function definition and schema
3. Simple examples (calculator, weather)
4. Advanced examples (database queries, API calls)
5. Best practices and error handling
6. Real-world applications

## Section 1: Environment Setup

Before we can call OpenAI, we need the client library and a way to load the API key securely from a `.env` file. This section sets up the imports and creates a reusable `OpenAI` client.

**Import the OpenAI SDK.**

`openai` provides the `OpenAI` client class that we use for every chat-completion request in this notebook.

In [1]:
import openai



**Import `python-dotenv` and `os`.**

`dotenv` reads key/value pairs from a `.env` file into environment variables, and `os` lets us read those values without hard-coding secrets.

In [2]:
from dotenv import load_dotenv
import os

**Load environment variables.**

`load_dotenv(override=True)` parses the `.env` file. The returned `True` confirms that a `.env` file was found and loaded.

In [3]:
load_dotenv(override=True)

True

**Create the OpenAI client.**

`client = openai.OpenAI(api_key=...)` initializes the client once. All later cells reuse this object, so we only need to configure the API key in one place.

In [4]:
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Section 2: Traditional Chat Completions (No Function Calling)

First, we use the chat API the "normal" way — with no tools. This shows what the model does when it must answer entirely from its own reasoning.

**Ask a math question the traditional way.**

We send `"What's 15 * 23 + 45?"` to `gpt-4o`. The model returns a text explanation and final answer, but it does not run any code.

In [5]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "What's 15 * 23 + 45?"}
    ]
)

print("Traditional Response:")
print(response.choices[0].message.content)

Traditional Response:
First, you calculate the multiplication:

\[ 15 \times 23 = 345 \]

Then, add 45 to the result:

\[ 345 + 45 = 390 \]

So, \( 15 \times 23 + 45 = 390 \).


**Create a reusable helper for plain chat.**

`get_open_ai_response(prompt)` wraps the API call with a system message ("GEN AI specialist"). This keeps the notebook concise and lets us test several prompts quickly.

In [6]:
def get_open_ai_response(prompt):
    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are specialized in GEN so answer the question like a GEN AI specialist"},
        {"role": "user", "content": prompt}
    ]
    )

    print("Traditional Response:")
    print(response.choices[0].message.content)


**Test the helper with a factual question.**

The model answers from its training data. Notice the disclaimer asking you to verify current facts — the model has a knowledge cutoff and can hallucinate dates.

In [7]:
get_open_ai_response("Who is PM of India")

Traditional Response:
As of my last update, the Prime Minister of India is Narendra Modi. He has been in office since May 26, 2014. However, please verify this information with a current source, as political positions can change due to elections or other political events.


**Run the exact same prompt again.**

The output is slightly different from the previous cell. This illustrates that LLM outputs are probabilistic, even with identical inputs.

In [8]:
get_open_ai_response("Who is PM of India")

Traditional Response:
As of my last update, the Prime Minister of India is Narendra Modi. He has been in office since May 26, 2014. However, please verify with the latest sources as political positions can change due to elections or other events.


**Test the helper with arithmetic.**

The model explains order-of-operations (PEMDAS) and gives the correct answer. It is still reasoning in text, not executing code.

In [9]:
get_open_ai_response("answer me 2+2*75")

Traditional Response:
When performing the calculation \(2 + 2 \times 75\), it's important to follow the order of operations, which in mathematics is often remembered by the acronym PEMDAS (Parentheses, Exponents, Multiplication and Division, Addition and Subtraction).

1. **Multiplication first:**  
   \(2 \times 75 = 150\)

2. **Then addition:**  
   \(2 + 150 = 152\)

So, the answer is \(152\).


**Redefine the helper with a mathematician persona.**

Changing the system message changes the style and depth of the answer. This is useful when you want the model to adopt a domain-specific tone.

In [10]:
def get_open_ai_response(prompt):
    response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are an expert AI mathematician with deep knowledge across all areas of mathematics, from elementary arithmetic to advanced topics like topology, abstract algebra, real analysis, complex analysis, differential equations, number theory, combinatorics, probability theory, statistics, discrete mathematics, linear algebra, and mathematical logic."},
        {"role": "user", "content": prompt}
    ],
    )
    

    print("Traditional Response:")
    print(response.choices[0].message.content)


**Run the same arithmetic with the new persona.**

The answer is now more formal and math-focused, but still produced entirely by the model.

In [11]:
get_open_ai_response("answer me 2+2*75")

Traditional Response:
To solve the expression \(2 + 2 \times 75\), we need to follow the order of operations, often remembered by the acronym PEMDAS (Parentheses, Exponents, Multiplication and Division (from left to right), Addition and Subtraction (from left to right)).

1. **Multiplication** comes before **Addition** in the order of operations. So, first we compute the multiplication:
   \[
   2 \times 75 = 150
   \]

2. Next, we proceed with the addition:
   \[
   2 + 150 = 152
   \]

Therefore, the result of the expression \(2 + 2 \times 75\) is \(152\).


## Section 3: Building a Local Calculator Tool

Function calling works by pairing a local Python function with a JSON schema. In this section we build the calculator function and describe it to OpenAI.

**Define and test a local `calculate` function.**

This Python function takes an operation and two numbers and returns the result. It also guards against division by zero and unsupported operators. We test it on `15 * 23` to make sure it works before exposing it to the model.

In [12]:
# Define a simple calculator function
def calculate(operation: str, a: float, b: float) -> float:
    """
    Perform basic mathematical operations
    
    Args:
        operation: The operation to perform (+, -, *, /)
        a: First number
        b: Second number
    
    Returns:
        Result of the calculation
    """
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        return a / b if b != 0 else "Error: Division by zero"
    else:
        return "Error: Unsupported operation"

# Test the function
result = calculate("*", 15, 23)
print(f"15 * 23 = {result}")

15 * 23 = 345


**Describe the calculator to OpenAI with a JSON schema.**

The schema tells the model:
- `name`: what the function is called
- `description`: when to use it
- `parameters`: the names, types, and allowed values of arguments
- `required`: which arguments must be supplied

This schema is what we will pass to the `functions` parameter of the chat API.

In [13]:
import json
# Define the function schema for OpenAI
calculator_function = {
    "name": "calculate",
    "description": "Perform basic mathematical operations like addition, subtraction, multiplication, and division",
    "parameters": {
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "description": "The mathematical operation to perform",
                "enum": ["+", "-", "*", "/"]
            },
            "a": {
                "type": "number",
                "description": "The first number"
            },
            "b": {
                "type": "number",
                "description": "The second number"
            }
        },
        "required": ["operation", "a", "b"]
    }
}

print("📋 Function schema defined!")
print(json.dumps(calculator_function, indent=2))

📋 Function schema defined!
{
  "name": "calculate",
  "description": "Perform basic mathematical operations like addition, subtraction, multiplication, and division",
  "parameters": {
    "type": "object",
    "properties": {
      "operation": {
        "type": "string",
        "description": "The mathematical operation to perform",
        "enum": [
          "+",
          "-",
          "*",
          "/"
        ]
      },
      "a": {
        "type": "number",
        "description": "The first number"
      },
      "b": {
        "type": "number",
        "description": "The second number"
      }
    },
    "required": [
      "operation",
      "a",
      "b"
    ]
  }
}


## Section 4: Basic Function Calling Implementation

Now we pass the calculator schema to the model. The model can choose to call `calculate` instead of answering in plain text. When it does, we run the local function and send the result back.

#### Basic Function Calling Implementation

**Implement a chat loop that can use the calculator.**

Line-by-line:
1. Build the message list with a system prompt and the user's question.
2. Call `client.chat.completions.create(..., functions=[...], function_call="auto")`. `auto` lets the model decide whether to call a function.
3. If the response contains `message.function_call`, extract the function name and arguments.
4. Execute the local `calculate` function with those arguments.
5. Append the function-call message and the function result to the conversation.
6. Send the updated conversation back to the model for a final, natural-language answer.

In [14]:
def chat_with_calculator(user_message: str):
    """
    Chat with OpenAI using function calling for calculations
    """
    messages = [
        {"role": "system", "content": "Yo are calculation specilist."},
        {"role": "user", "content": user_message}
    ]
    
    # Make the initial request with function definition
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        functions=[calculator_function],
        function_call="auto"  # Let OpenAI decide when to call functions
    )
    
    message = response.choices[0].message
    if message.function_call:
        function_name = message.function_call.name
        function_args = json.loads(message.function_call.arguments)
        
        print(f"🔧 OpenAI wants to call: {function_name}")
        print(f"📝 Arguments: {function_args}")
        if function_name == "calculate":
            result = calculate(**function_args)

            messages.append({
                "role": "assistant",
                "content": None,
                "function_call": {
                    "name": function_name,
                    "arguments": json.dumps(function_args)
                }
            })

            messages.append({
                "role": "function",
                "name": function_name,
                "content": str(result)
            })
            
            # Get final response from OpenAI
            final_response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages
            )
        return final_response.choices[0].message.content



        



**Test with a non-math question.**

The model does not need the calculator for `"who is the PM of India"`, so it should not call a function. In the current implementation, the helper returns `None` when no function is called.

In [15]:
# Test the function calling
result = chat_with_calculator("who is the PM of India")
print("🎯 Function Calling Result:")
print(result)

🎯 Function Calling Result:
None


**Test with a simple multiplication.**

Again, the printed result is `None` because the current helper returns a value only when a function is called. The variable still receives the final answer if a function call occurs.

In [16]:
# Test the function calling
result = chat_with_calculator("What's 15 * 23? Please calculate step by step.")
print("🎯 Function Calling Result:")
print(result)

🎯 Function Calling Result:
None


**Test with a multi-step expression.**

Here the model decides to call `calculate` with `operation='*', a=15, b=23`. You can see the function name and arguments printed. This particular helper calls the tool once per turn, so the returned value reflects the intermediate result.

In [17]:
result = chat_with_calculator("What's 15 * 23 + 45? Please calculate step by step.")


🔧 OpenAI wants to call: calculate
📝 Arguments: {'operation': '*', 'a': 15, 'b': 23}


**Print the final answer from the previous turn.**

After the tool result is sent back to the API, the model synthesizes a friendly, step-by-step final answer: 390.

In [18]:
print(result)

First, calculate the product of 15 and 23:

\[ 15 \times 23 = 345 \]

Next, add 45 to the result:

\[ 345 + 45 = 390 \]

So, \( 15 \times 23 + 45 = 390 \).


## Section 5: Adding a Weather Tool

Real-world tools often have optional parameters and return structured text. We create a mock weather lookup and its schema so the model can ask for current conditions.

**Define `get_weather` and its schema.**

`get_weather` uses a small dictionary of mock data for three cities. In production you would replace this with a request to a live weather API. The schema makes `city` required and `country` optional, with a default of `"US"`.

In [19]:
# More complex example - Weather API
def get_weather(city: str, country: str = "US") -> str:
    """
    Get current weather for a city
    
    Args:
        city: Name of the city
        country: Country code (default: US)
    
    Returns:
        Weather information as string
    """
    # Using a mock weather API for demonstration
    # In real implementation, you'd use actual weather API
    mock_weather_data = {
        "new york": {"temp": 22, "condition": "sunny", "humidity": 60},
        "london": {"temp": 15, "condition": "cloudy", "humidity": 80},
        "tokyo": {"temp": 28, "condition": "rainy", "humidity": 75}
    }
    
    city_key = city.lower()
    if city_key in mock_weather_data:
        data = mock_weather_data[city_key]
        return f"Weather in {city}: {data['temp']}°C, {data['condition']}, humidity: {data['humidity']}%"
    else:
        return f"Weather data not available for {city}"

# Define weather function schema
weather_function = {
    "name": "get_weather",
    "description": "Get current weather information for a specific city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Name of the city"
            },
            "country": {
                "type": "string",
                "description": "Country code (e.g., US, UK, JP)",
                "default": "US"
            }
        },
        "required": ["city"]
    }
}

print("🌤️ Weather function ready!")

🌤️ Weather function ready!


**Placeholder cells.**

The next three code cells are intentionally empty. Use them to experiment, for example:

```python
get_weather("london")
handler.chat("Do I need an umbrella in Tokyo?")
```

## Section 6: Handling Multiple Functions

Most agents need more than one tool. We now give the model access to both `calculate` and `get_weather`, and let it choose the right one for each prompt.

###  Multiple Functions Handler

**Build a generic `FunctionHandler` class.**

The class keeps two things in sync:
- `self.functions`: a dictionary mapping function names to the actual Python functions
- `self.function_schemas`: the list of schemas passed to the API

The `chat` method:
1. Sends the user message and schemas.
2. Reads which function the model wants to call.
3. Executes the matching function locally.
4. Sends the result back and returns a final answer.

In [20]:
# Create a handler that can work with multiple functions
class FunctionHandler:
    def __init__(self):
        self.functions = {
            "calculate": calculate,
            "get_weather": get_weather
        }
        
        self.function_schemas = [
            calculator_function,
            weather_function
        ]
    
    def call_function(self, function_name: str, arguments: dict):
        """Call the appropriate function with given arguments"""
        if function_name in self.functions:
            return self.functions[function_name](**arguments)
        else:
            return f"Function {function_name} not found"
    
    def chat(self, user_message: str):
        """Enhanced chat with multiple function support"""
        messages = [{"role": "user", "content": user_message}]
        
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            functions=self.function_schemas,
            function_call="auto"
        )
        
        message = response.choices[0].message
        
        if message.function_call:
            function_name = message.function_call.name
            function_args = json.loads(message.function_call.arguments)
            
            print(f"🔧 Calling function: {function_name}")
            print(f"📝 Arguments: {function_args}")
            
            # Execute the function
            result = self.call_function(function_name, function_args)
            
            # Continue conversation with function result
            messages.extend([
                {
                    "role": "assistant",
                    "content": None,
                    "function_call": {
                        "name": function_name,
                        "arguments": json.dumps(function_args)
                    }
                },
                {
                    "role": "function",
                    "name": function_name,
                    "content": str(result)
                }
            ])
            
            final_response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages
            )
            
            return final_response.choices[0].message.content
        
        return message.content

# Initialize the handler
handler = FunctionHandler()
print("🤖 Multi-function handler ready!")

🤖 Multi-function handler ready!


**Weather request: Tokyo.**

The model chooses `get_weather`, extracts `city='Tokyo'` and `country='JP'`, and the handler returns a natural-language summary using the mock data.

In [21]:
handler.chat("what is the Temperature Tokyo")

🔧 Calling function: get_weather
📝 Arguments: {'city': 'Tokyo', 'country': 'JP'}


'The current temperature in Tokyo is 28°C, with rainy conditions and a humidity of 75%.'

**Math request: 2 + 2.**

The model chooses `calculate` this time. The handler runs `calculate(operation='+', a=2, b=2)` and returns the final answer.

In [22]:
handler.chat("what is the result of 2+2")

🔧 Calling function: calculate
📝 Arguments: {'operation': '+', 'a': 2, 'b': 2}


'The result of 2 + 2 is 4.'

**Experimentation cell.**

Try your own prompts here. Good candidates:
- `"What is 100 / 4 and is it sunny in London?"`
- `"If it is 22°C in New York and 15°C in London, which city is warmer and by how much?"`

## Summary

In this notebook we:

1. Set up the OpenAI client and API key.
2. Saw how traditional chat completions work without tools.
3. Built local tools (`calculate`, `get_weather`) and their JSON schemas.
4. Used `functions` and `function_call="auto"` to let the model decide when to call a tool.
5. Handled single and multiple tool calls with a reusable `FunctionHandler`.

Function calling is the bridge between natural language and real-world actions. By describing tools clearly and routing the model's choices back to your code, you can turn ChatGPT into an agent that calculates, queries APIs, and interacts with databases.